# 03 — Structured Outputs

## Objectives

- Get a **validated Python object** out of an agent instead of a string
- Offer the model a **choice** of output types (unions)
- Enforce your own rules with `output_validator` + `ModelRetry`
- Understand what the default *ToolOutput* mode does under the hood

In [ ]:
# ── SETUP — run this first in every notebook ─────────────────────────────────
import os
import sys

import logfire
from dotenv import find_dotenv, load_dotenv

# 1) Environment: walk up from notebooks/ to the repo root and load .env
load_dotenv(find_dotenv())

# 2) Observability: no Logfire account yet? This still works — data only ships
#    when a token exists (`uv run logfire auth`, or a LOGFIRE_TOKEN in .env).
logfire.configure(send_to_logfire="if-token-present")
logfire.instrument_pydantic_ai()  # v2: instrumentation is explicit — Agent(instrument=True) is gone

MODEL = os.getenv("MODEL", "openai:gpt-6.1-sol")

print("kernel :", sys.executable)  # should end in learningdantic/.venv/bin/python
print("MODEL  :", MODEL)

## The idea

You define a Pydantic model; the agent's `output_type` becomes exactly that. This is the "pydantic"
in pydantic-ai: no regex-parsing JSON blobs out of chat strings.

Under the default **ToolOutput** mode, pydantic-ai quietly adds a hidden "final result" tool to the
model, forces the model to call it with JSON, and validates that JSON into your class. (Two other
modes exist — `NativeOutput` for providers with native JSON output, `PromptedOutput` for models
without tool support — but ToolOutput is the default and the one you'll want.)

In [ ]:
from pydantic import BaseModel
from pydantic_ai import Agent


class Hero(BaseModel):
    name: str
    power: str
    weakness: str


hero_agent = Agent(
    MODEL,
    output_type=Hero,
    instructions="Extract the hero profile from the user text. Invent nothing.",
)

result = await hero_agent.run(
    "Captain Cardboard — his power is folding into any shape; he loses strength near water."
)
hero = result.output  # a real Hero instance, validated by Pydantic
print(hero)
print("it is a real object:", type(hero).__name__, "| name.upper() =", hero.name.upper())

> ✅ **Checkpoint** — `result.output` is a `Hero`, not a string. Autocomplete works on it,
> type checkers understand it, and an invalid extraction would have *raised*, not returned junk.

## Union outputs — let the model choose a shape

A **list** of output types gives the model one output tool per type; it picks whichever fits.
`result.output` is an instance of whichever it chose — branch with `isinstance`. (This exact
mechanism is the seed of multi-agent routing in Phase 2.)

In [ ]:
class Fact(BaseModel):
    fact: str


class Question(BaseModel):
    question: str


sorter = Agent(
    MODEL,
    output_type=[Fact, Question],
    instructions="If the user states something, record it as a Fact; if they ask, produce a Question.",
)

r1 = await sorter.run("The Eiffel Tower is about 330 meters tall.")
r2 = await sorter.run("How tall is the Eiffel Tower?")

print("r1 →", type(r1.output).__name__, ":", r1.output)
print("r2 →", type(r2.output).__name__, ":", r2.output)

> ✅ **Checkpoint** — one run produced a `Fact`, the other a `Question`, from the same agent.
> Now your downstream code can `isinstance(r.output, Fact)` instead of guessing.

## `output_validator` + `ModelRetry` — rules the model can't ignore

Pydantic validates *shape* (types, required fields). For *meaning* — business rules — you add an
`output_validator`. Raising `ModelRetry` sends your rejection back to the model with the error
message, so it tries again. The run only finishes with output you accept.

In [ ]:
from pydantic_ai import ModelRetry


@hero_agent.output_validator
def shout_the_name(ctx, output: Hero) -> Hero:
    """The registry requires hero names in ALL CAPS."""
    if not output.name.isupper():
        raise ModelRetry(f"The name must be ALL CAPS — {output.name!r} is not. Regenerate.")
    return output


result = await hero_agent.run(
    "Captain Cardboard — his power is folding into any shape; he loses strength near water."
)
print(result.output)

> ✅ **Checkpoint** — the name came back in ALL CAPS. The model almost certainly tried
`Captain Cardboard` first and got bounced — that's the `ModelRetry` loop working.
>
> 🔭 In Logfire, expand this run's trace: you can literally find the rejected first attempt and
> the retry. This feedback loop is *the* reason structured outputs are reliable enough to build on.

## 🏋️ Exercise — the recipe extractor

Define a `Recipe` model — `title: str`, `ingredients: list[str]`, `steps: list[str]`,
`minutes: int` — and an agent that extracts a recipe from any text you paste. Add an
`output_validator` that uses `ModelRetry` to reject recipes with **empty ingredients** or
`minutes <= 0`. Run it on a recipe you find online. Bonus: make it reject recipes with
more than 10 ingredients with a friendlier message.

Your take: [`../solutions/03_recipe_extractor.ipynb`](../solutions/03_recipe_extractor.ipynb).

## Next

→ **[04_tools.ipynb](04_tools.ipynb)** — the model calls *your* code.